In [0]:
!pip install hotel_reservation-0.0.1-py3-none-any.whl

Processing ./hotel_reservation-0.0.1-py3-none-any.whl
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/91.2 kB ? eta -:--:--
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 6.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/61.9 kB ? eta -:--:--
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/68.0 kB ? eta -:--:--
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.0/68.0 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/52.4 kB ? eta -:--:--
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/57.7 kB ? eta -:--:--
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 6.0 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of mlflow-skinny[databricks] to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looki

In [0]:
%restart_python

In [0]:
"""
Ray Tune + MLflow Nested Runs for Hotel Reservation Prediction
"""
from datetime import datetime

import mlflow
import pandas as pd
from pyspark.sql import SparkSession
from ray import tune
from ray.tune.search.optuna import OptunaSearch
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.model_selection import train_test_split

from hotel_reservation.config import ProjectConfig, Tags
from hotel_reservation.models.basic_model import BasicModel

In [0]:
config = ProjectConfig.from_yaml(config_path="../project_config.yml", env="dev")
tags = Tags(**{"git_sha": "abcd12345", "branch": "week3"})
spark = SparkSession.builder.getOrCreate()

basic_model = BasicModel(config=config, tags=tags, spark=spark)
basic_model.load_data()

# Split train set into train/validation for tuning

X_train, X_valid, y_train, y_valid = train_test_split(
    basic_model.X_train, basic_model.y_train_encoded, test_size=0.2, random_state=42
)

2025-09-22 12:23:53.070 | INFO     | hotel_reservation.models.basic_model:load_data:61 - 🔄 Loading data from Databricks tables...
2025-09-22 12:24:16.639 | INFO     | hotel_reservation.models.basic_model:load_data:71 - ✅ Data successfully loaded.
2025-09-22 12:24:16.650 | INFO     | hotel_reservation.models.basic_model:load_data:76 - ✅ Target successfully encoded.


In [0]:
# --- Trainable function for Ray Tune with nested MLflow runs ---
""" This is the function that Ray Tune will call for each hyperparameter combination. 
For each trial:
1.Start a nested MLflow run (grouped under a parent run for easy tracking).
2.Update the model’s hyperparameters for this trial.
3.Prepare the pipeline using BasicModel.prepare_features().
4.Train the model on the training set.
5.Predict on the validation set.
6.Compute metrics: Accuracy, Precision, Recall.
7.Log parameters and metrics to MLflow.
8.Report metrics back to Ray Tune for optimization."""


def train_with_nested_mlflow(
    config,
    X_train_in: pd.DataFrame,
    X_valid_in: pd.DataFrame,
    y_train_in: pd.DataFrame,
    y_valid_in: pd.DataFrame,
    project_config: ProjectConfig,
    experiment_id: str,
    parent_run_id: str,
):
    n_estimators, max_depth, learning_rate = (
        config["n_estimators"],
        config["max_depth"],
        config["learning_rate"],
    )
    mlflow.enable_system_metrics_logging()
    with mlflow.start_run(
        run_name=f"trial_n{n_estimators}_md{max_depth}_lr{learning_rate}",
        nested=True,
        parent_run_id=parent_run_id,
        experiment_id=experiment_id,
    ):
        # Update parameters for this trial
        trial_params = dict(project_config.parameters)
        trial_params.update(
            {
                "n_estimators": n_estimators,
                "max_depth": max_depth,
                "learning_rate": learning_rate,
            }
        )

        # Train model
        model = BasicModel(config=project_config, tags=tags, spark=None)
        model.parameters = trial_params
        model.prepare_features()
        model.pipeline.set_params(
            classifier__n_estimators=n_estimators,
            classifier__max_depth=max_depth,
            classifier__learning_rate=learning_rate,
        )
        model.pipeline.fit(X_train_in, y_train_in)

        y_pred = model.pipeline.predict(X_valid_in)
        metrics = {
            "accuracy": accuracy_score(y_valid_in, y_pred),
            "precision": precision_score(y_valid_in, y_pred),
            "recall": recall_score(y_valid_in, y_pred),
        }
        mlflow.log_params(config)
        mlflow.log_metrics(metrics)
        tune.report(metrics)

In [0]:
def define_by_run_func(trial):
    trial.suggest_int("n_estimators", 100, 600, log=True)
    trial.suggest_int("max_depth", 3, 15)
    trial.suggest_float("learning_rate", 0.01, 0.2)


# Define Optuna search algo
algo = OptunaSearch(space=define_by_run_func, metric="recall", mode="max")
# Note: A concurrency limiter, limits the number of parallel trials. This is important for Bayesian search (inherently sequential) as too many parallel trials reduces the benefits of priors to inform the next search round.
# algo = ConcurrencyLimiter(algo, max_concurrent=num_cpu_cores_per_worker*max_worker_nodes+num_cpus_head_node)

[I 2025-09-22 12:31:14,428] A new study created in memory with name: optuna


In [0]:
# --- Launch Ray Tune experiment with MLflow parent run ---
import os

import ray
from mlflow.utils.databricks_utils import get_databricks_env_vars

mlflow_dbrx_creds = get_databricks_env_vars("databricks")
os.environ["DATABRICKS_HOST"] = mlflow_dbrx_creds["DATABRICKS_HOST"]
os.environ["DATABRICKS_TOKEN"] = mlflow_dbrx_creds["DATABRICKS_TOKEN"]

# for distributed, use this:

# ray_conf = setup_ray_cluster(
#   min_worker_nodes=2,
#   max_worker_nodes=2,
#   num_cpus_head_node=1,
#   num_cpus_worker_node=2,
# )
# os.environ['RAY_ADDRESS'] = ray_conf[0]

n_trials = 20

mlflow.enable_system_metrics_logging()
mlflow.set_experiment("/Shared/hotel-reservation-finetuning")
experiment_id = mlflow.get_experiment_by_name("/Shared/hotel-reservation-finetuning").experiment_id
with mlflow.start_run(
    run_name=f"optuna-finetuning-{datetime.now().strftime('%Y-%m-%d')}",
    tags={"git_sha": "1234567890abcd", "branch": "main"},
    description="LightGBM hyperparameter tuning with Ray & Optuna",
) as parent_run:
    tuner = tune.Tuner(
        ray.tune.with_parameters(
            train_with_nested_mlflow,
            X_train_in=X_train,
            y_train_in=y_train,
            X_valid_in=X_valid,
            y_valid_in=y_valid,
            project_config=config,
            parent_run_id=parent_run.info.run_id,
            experiment_id=experiment_id,
        ),
        tune_config=tune.TuneConfig(
            search_alg=algo,
            num_samples=n_trials,
            reuse_actors=True,  # Highly recommended for short training jobs (NOT RECOMMENDED FOR GPU AND LONG TRAINING JOBS)
        ),
        # run_config=train.RunConfig(
        #     name="ray-tune-optuna",
        #     callbacks=[
        #         MLflowLoggerCallback(
        #             experiment_name=experiment_name,
        #             save_artifact=False,
        #             tags={"mlflow.parentRunId": parent_run.info.run_id})]
        # )
    )
    results = tuner.fit()

# --- Retrieve best parameters ---
best_result = results.get_best_result(metric="recall", mode="max")
print("Best hyperparameters:", best_result.config)

2025/09/22 12:34:01 INFO mlflow.tracking.fluent: Experiment with name '/Shared/hotel-reservation-finetuning' does not exist. Creating a new experiment.
2025/09/22 12:34:02 INFO mlflow.system_metrics.system_metrics_monitor: Skip logging GPU metrics. Set logger level to DEBUG for more details.
2025/09/22 12:34:02 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.
2025-09-22 12:34:04,666	INFO worker.py:1917 -- Started a local Ray instance.
2025-09-22 12:34:05,991	INFO tune.py:253 -- Initializing Ray automatically. For cluster usage or custom Ray initialization, call `ray.init(...)` before `Tuner(...)`.


+---------------------------------------------------------------------------------+
| Configuration for experiment     train_with_nested_mlflow_2025-09-22_12-34-02   |
+---------------------------------------------------------------------------------+
| Search algorithm                 SearchGenerator                                |
| Scheduler                        FIFOScheduler                                  |
| Number of trials                 20                                             |
+---------------------------------------------------------------------------------+

View detailed results here: /root/ray_results/train_with_nested_mlflow_2025-09-22_12-34-02
To visualize your results with TensorBoard, run: `tensorboard --logdir /local_disk0/tmp/ray/session_2025-09-22_12-34-02_467730_1802/artifacts/2025-09-22_12-34-06/train_with_nested_mlflow_2025-09-22_12-34-02/driver_artifacts`

Trial status: 1 PENDING
Current time: 2025-09-22 12:34:06. Total running time: 0s
Logical reso

(train_with_nested_mlflow pid=4522) Mon Sep 22 12:34:11 2025 Connection to spark from PID  4522
(train_with_nested_mlflow pid=4522) Mon Sep 22 12:34:11 2025 Initialized gateway on port 44205
(train_with_nested_mlflow pid=4522) Mon Sep 22 12:34:11 2025 Connected to spark.
(train_with_nested_mlflow pid=4522) 2025/09/22 12:34:12 INFO mlflow.system_metrics.system_metrics_monitor: Skip logging GPU metrics. Set logger level to DEBUG for more details.
(train_with_nested_mlflow pid=4522) 2025/09/22 12:34:12 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.
(train_with_nested_mlflow pid=4522) 2025-09-22 12:34:12.379 | INFO     | hotel_reservation.models.basic_model:prepare_features:84 - 🔄 Defining preprocessing pipeline...
(train_with_nested_mlflow pid=4522) 2025-09-22 12:34:12.379 | INFO     | hotel_reservation.models.basic_model:prepare_features:92 - ✅ Preprocessing pipeline defined.


(train_with_nested_mlflow pid=4522) [LightGBM] [Info] Number of positive: 15607, number of negative: 7609
(train_with_nested_mlflow pid=4522) [LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004766 seconds.
(train_with_nested_mlflow pid=4522) You can set `force_row_wise=true` to remove the overhead.
(train_with_nested_mlflow pid=4522) And if memory is not enough, you can set `force_col_wise=true`.
(train_with_nested_mlflow pid=4522) [LightGBM] [Info] Total Bins 656
(train_with_nested_mlflow pid=4522) [LightGBM] [Info] Number of data points in the train set: 23216, number of used features: 28
(train_with_nested_mlflow pid=4522) [LightGBM] [Info] [binary:BoostFromScore]: pavg=0.672252 -> initscore=0.718388
(train_with_nested_mlflow pid=4522) [LightGBM] [Info] Start training from score 0.718388


(train_with_nested_mlflow pid=4522) /local_disk0/.ephemeral_nfs/envs/pythonEnv-2b372688-b2e5-4fa8-8471-3f27971ddbb6/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
(train_with_nested_mlflow pid=4522)   warnings.warn(


...

(train_with_nested_mlflow pid=4522) [LightGBM] [Info] Number of positive: 15607, number of negative: 7609
(train_with_nested_mlflow pid=4522) [LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004631 seconds.
(train_with_nested_mlflow pid=4522) You can set `force_row_wise=true` to remove the overhead.
(train_with_nested_mlflow pid=4522) And if memory is not enough, you can set `force_col_wise=true`.
(train_with_nested_mlflow pid=4522) [LightGBM] [Info] Total Bins 656
(train_with_nested_mlflow pid=4522) [LightGBM] [Info] Number of data points in the train set: 23216, number of used features: 28
(train_with_nested_mlflow pid=4522) [LightGBM] [Info] [binary:BoostFromScore]: pavg=0.672252 -> initscore=0.718388
(train_with_nested_mlflow pid=4522) [LightGBM] [Info] Start training from score 0.718388


(train_with_nested_mlflow pid=4522) /local_disk0/.ephemeral_nfs/envs/pythonEnv-2b372688-b2e5-4fa8-8471-3f27971ddbb6/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
(train_with_nested_mlflow pid=4522)   warnings.warn(
(train_with_nested_mlflow pid=4522) 2025/09/22 12:34:54 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...
2025-09-22 12:34:55,045	INFO tune.py:1009 -- Wrote the latest version of all result files and experiment state to '/root/ray_results/train_with_nested_mlflow_2025-09-22_12-34-02' in 0.0174s.


(train_with_nested_mlflow pid=4522) 🏃 View run trial_n306_md13_lr0.06697268163528836 at: https://dbc-f122dc18-1b68.cloud.databricks.com/ml/experiments/1254421901833913/runs/5e708286a68d4c4d8818a7c4ee25483a
(train_with_nested_mlflow pid=4522) 🧪 View experiment at: https://dbc-f122dc18-1b68.cloud.databricks.com/ml/experiments/1254421901833913

Trial train_with_nested_mlflow_54b56b0f completed after 1 iterations at 2025-09-22 12:34:55. Total running time: 48s
+------------------------------------------------------------+
| Trial train_with_nested_mlflow_54b56b0f result             |
+------------------------------------------------------------+
| checkpoint_dir_name                                        |
| time_this_iter_s                                   1.50878 |
| time_total_s                                       1.50878 |
| training_iteration                                       1 |
| accuracy                                           0.89197 |
| precision                        

(train_with_nested_mlflow pid=4522) 2025/09/22 12:34:55 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!
2025/09/22 12:34:56 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...
2025/09/22 12:34:56 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!



Best hyperparameters: {'n_estimators': 310, 'max_depth': 15, 'learning_rate': 0.11992656858252698}



Reference docs:
- https://docs.ray.io/en/latest/index.html$0
- https://github.com/databricks-industry-solutions/ray-framework-on-databricks/blob/main/Hyperparam_Optimization/1-HPO-ML-Training-Optuna/01_hpo_optuna_ray_train.py#L502$0
- https://docs.databricks.com/aws/en/machine-learning/ray/ray-create$0
- https://docs.ray.io/en/latest/tune/key-concepts.html$0
